# 02 — Comparaison baseline vs improved sur RSNA dev

Ce notebook compare les deux variantes du prédicteur sur les **150 cas dev RSNA réels**.

| Variante | Modèle | Prompt | Comportement |
|----------|--------|--------|--------------|
| `baseline_v1` | MedGemma-4B (fallback : règles) | `baseline_prompt.txt` | Normal / Opacité / Incertain |
| `improved_v1` | MedGemma-4B (fallback : règles conservateur) | `improved_prompt.txt` | Idem + abstention si confiance < 0.60 ou marge ambiguë |

> **Ce qu'on mesure ici** : est-ce que le prompt renforcé réduit les faux positifs en s'abstenant sur les cas ambigus, au prix d'un taux d'incertitude plus élevé ?

In [ ]:
import sys, os, json, csv, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.inference import baseline_predict, improved_predict
from src.guardrails import apply_safety_guardrails
from src.metrics import macro_f1

CASES_CSV = ROOT / 'data' / 'rsna' / 'cases.csv'
OUT_DIR   = ROOT / 'eval' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CASES_CSV, newline='', encoding='utf-8') as f:
    dev_cases = [r for r in csv.DictReader(f) if r['split'] == 'dev']

print(f'USE_MEDGEMMA : {os.getenv("USE_MEDGEMMA","0")}')
print(f'Dev cases    : {len(dev_cases)}')

## 1. Inférence — baseline et improved sur les 150 cas

In [ ]:
def run_predictor(cases, predictor, label):
    rows = []
    t0 = time.perf_counter()
    for i, case in enumerate(cases):
        img_path = ROOT / case['image_path']
        pred = apply_safety_guardrails(predictor(img_path))
        rows.append({
            'case_id'        : case['case_id'],
            'label'          : case['label'],
            'predicted_class': pred['predicted_class'],
            'confidence'     : pred['confidence'],
            'image_quality'  : pred['image_quality'],
            'latency_ms'     : pred['latency_ms'],
            'model_name'     : pred['model_name'],
            'correct'        : case['label'] == pred['predicted_class'],
        })
        if (i + 1) % 50 == 0:
            print(f'  {label} [{i+1}/150]')
    elapsed = time.perf_counter() - t0
    print(f'  {label} terminé en {elapsed:.1f}s')
    return pd.DataFrame(rows)

print('=== Baseline ===')
df_base = run_predictor(dev_cases, baseline_predict, 'baseline')

print('\n=== Improved ===')
df_imp  = run_predictor(dev_cases, improved_predict, 'improved')

## 2. Métriques comparées

In [ ]:
def compute_metrics(df, name):
    y_true = df['label'].tolist()
    y_pred = df['predicted_class'].tolist()
    n = len(y_true)

    accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / n
    f1 = macro_f1(y_true, y_pred)

    tp_o = sum(t == 'suspected_opacity' and p == 'suspected_opacity' for t, p in zip(y_true, y_pred))
    fn_o = sum(t == 'suspected_opacity' and p != 'suspected_opacity' for t, p in zip(y_true, y_pred))
    sensitivity = tp_o / (tp_o + fn_o) if (tp_o + fn_o) else 0

    tn_n = sum(t == 'normal' and p == 'normal' for t, p in zip(y_true, y_pred))
    fp_n = sum(t == 'normal' and p != 'normal' for t, p in zip(y_true, y_pred))
    specificity = tn_n / (tn_n + fp_n) if (tn_n + fp_n) else 0

    uncertain_rate = sum(p == 'uncertain' for p in y_pred) / n

    # Faux positifs opacité (image normale prédite opaque)
    fp_opac = sum(t == 'normal' and p == 'suspected_opacity' for t, p in zip(y_true, y_pred))
    # Faux négatifs opacité (opacité prédite normale)
    fn_opac = sum(t == 'suspected_opacity' and p == 'normal' for t, p in zip(y_true, y_pred))

    return {
        'variante'          : name,
        'accuracy'          : round(accuracy, 4),
        'macro_f1'          : round(f1, 4),
        'sensibilité_opacité': round(sensitivity, 4),
        'spécificité_normal': round(specificity, 4),
        'taux_incertitude'  : round(uncertain_rate, 4),
        'FP_opacité'        : fp_opac,
        'FN_opacité'        : fn_opac,
        'latence_médiane_ms': float(np.median(df['latency_ms'])),
    }

m_base = compute_metrics(df_base, 'baseline_v1')
m_imp  = compute_metrics(df_imp,  'improved_v1')

df_metrics = pd.DataFrame([m_base, m_imp]).set_index('variante')
print('=== COMPARAISON BASELINE vs IMPROVED — DEV RSNA (150 cas) ===')
print(df_metrics.T.to_string())

## 3. Visualisations comparées

In [ ]:
CLASSES = ['normal', 'suspected_opacity', 'uncertain']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution des prédictions
base_counts = [sum(df_base['predicted_class'] == c) for c in CLASSES]
imp_counts  = [sum(df_imp['predicted_class']  == c) for c in CLASSES]
gt_counts   = [sum(df_base['label'] == c) for c in CLASSES]
x = np.arange(3); w = 0.25
axes[0,0].bar(x - w, gt_counts,   w, label='Ground truth', color='steelblue', alpha=0.85)
axes[0,0].bar(x,     base_counts, w, label='Baseline',     color='coral',     alpha=0.85)
axes[0,0].bar(x + w, imp_counts,  w, label='Improved',     color='seagreen',  alpha=0.85)
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(['normal', 'opacity', 'uncertain'])
axes[0,0].set_title('Distribution des prédictions'); axes[0,0].legend(); axes[0,0].grid(axis='y', alpha=0.3)

# 2. Métriques clés
metric_keys  = ['accuracy', 'macro_f1', 'sensibilité_opacité', 'spécificité_normal', 'taux_incertitude']
metric_short = ['Accuracy', 'Macro-F1', 'Sensib. opac.', 'Specif. norm.', 'Taux incert.']
vals_base = [m_base[k] for k in metric_keys]
vals_imp  = [m_imp[k]  for k in metric_keys]
x2 = np.arange(len(metric_keys)); w2 = 0.35
axes[0,1].bar(x2 - w2/2, vals_base, w2, label='Baseline', color='coral',    alpha=0.85)
axes[0,1].bar(x2 + w2/2, vals_imp,  w2, label='Improved', color='seagreen', alpha=0.85)
axes[0,1].set_xticks(x2); axes[0,1].set_xticklabels(metric_short, rotation=15, ha='right')
axes[0,1].set_ylim(0, 1.1); axes[0,1].set_title('Métriques clés')
axes[0,1].legend(); axes[0,1].grid(axis='y', alpha=0.3)
for i, (vb, vi) in enumerate(zip(vals_base, vals_imp)):
    axes[0,1].text(i - w2/2, vb + 0.02, f'{vb:.2f}', ha='center', fontsize=8)
    axes[0,1].text(i + w2/2, vi + 0.02, f'{vi:.2f}', ha='center', fontsize=8)

# 3. Cas où les deux modèles divergent
disagree = df_base['predicted_class'] != df_imp['predicted_class']
n_disagree = disagree.sum()
df_disagree = df_base[disagree].copy()
df_disagree['improved_pred'] = df_imp[disagree]['predicted_class'].values
# Compter les types de désaccord
disagree_pairs = (df_disagree['predicted_class'] + ' → ' + df_disagree['improved_pred']).value_counts()
axes[1,0].barh(disagree_pairs.index, disagree_pairs.values, color='orchid', edgecolor='white')
axes[1,0].set_title(f'Désaccords baseline→improved ({n_disagree} cas)')
axes[1,0].set_xlabel('Nombre de cas'); axes[1,0].grid(axis='x', alpha=0.3)

# 4. Confiance : baseline vs improved
axes[1,1].scatter(df_base['confidence'], df_imp['confidence'],
                  c=df_base['correct'].map({True:'seagreen', False:'coral'}),
                  alpha=0.5, s=30)
axes[1,1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1,1].set_xlabel('Confiance baseline'); axes[1,1].set_ylabel('Confiance improved')
axes[1,1].set_title('Confiance baseline vs improved\n(vert=correct, rouge=erreur)')
axes[1,1].set_xlim(0, 1); axes[1,1].set_ylim(0, 1)
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'comparison_baseline_improved.png'), dpi=120, bbox_inches='tight')
plt.show()

## 4. Cas de désaccord — analyse détaillée

In [ ]:
print(f'Cas où baseline ≠ improved : {n_disagree}/150')
print('\nDétail des désaccords (ground truth | baseline | improved) :')
for _, row in df_disagree.iterrows():
    gt = row['label']
    bp = row['predicted_class']
    ip = row['improved_pred']
    # Qui a raison ?
    winner = 'baseline' if bp == gt else ('improved' if ip == gt else 'aucun')
    print(f'  {row["case_id"][:30]}  GT={gt:<20} Base={bp:<20} Imp={ip:<20} → {winner}')

## 5. Sauvegarde

In [ ]:
# CSVs prédictions
df_base.to_csv(OUT_DIR / 'rsna_dev_baseline_predictions.csv', index=False)
df_imp.to_csv(OUT_DIR  / 'rsna_dev_improved_predictions.csv', index=False)

# Summary JSON
summary = [m_base, m_imp]
with open(OUT_DIR / 'rsna_dev_comparison_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('Fichiers sauvegardés :')
for p in sorted(OUT_DIR.glob('rsna_dev*')):
    print(f'  {p.name}')

print('\n=== TABLEAU FINAL ===')
delta = {k: round(m_imp[k] - m_base[k], 4) for k in ['accuracy','macro_f1','sensibilité_opacité','spécificité_normal','taux_incertitude'] if isinstance(m_base[k], float)}
print(pd.DataFrame({'baseline': m_base, 'improved': m_imp, 'delta': {**delta}}).T[['accuracy','macro_f1','sensibilité_opacité','spécificité_normal','taux_incertitude']].to_string())